Q1. When generating questions for the first 3 lesson pages, what is the average number of input tokens across these 3 calls? (1 point) ANS: ~1400

In [3]:
from gitsource import GithubRepositoryDataReader
from openai import OpenAI
import json
from dotenv import load_dotenv

load_dotenv()
openai_client = OpenAI()

# 1. Load all lesson pages from the course repository
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

# 2. Filter to get only lessons from the first module (01-agentic-rag)
module1_lessons = []
for doc in documents:
    if "01-agentic-rag/lessons/" in doc["filename"]:
        module1_lessons.append(doc)

# 3. Sort by filename to get the first 3 lessons
module1_lessons.sort(key=lambda x: x["filename"])
first_3_lessons = module1_lessons[:3]

print("First 3 lesson pages:")
for i, doc in enumerate(first_3_lessons, 1):
    print(f"{i}. {doc['filename']}")
    print(f"   Content preview: {doc['content'][:100]}...")
    print()

# 4. Define the data generation instructions (same as in your code)
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

First 3 lesson pages:
1. 01-agentic-rag/lessons/01-intro.md
   Content preview: # Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxU...

2. 01-agentic-rag/lessons/02-environment.md
   Content preview: # Environment

Video: [Watch this lesson](https://www.youtube.com/watch?v=3U4gBrmkZyM&list=PL3MmuxUb...

3. 01-agentic-rag/lessons/03-rag.md
   Content preview: # RAG

Video: [Watch this lesson](https://www.youtube.com/watch?v=JktYwBIDErk&list=PL3MmuxUbc_hLZFNg...



In [4]:
tokens_per_page_api = []

for doc in first_3_lessons:
    messages = [
        {"role": "developer", "content": data_gen_instructions},
        {"role": "user", "content": doc["content"]}
    ]
    
    # Use the API to count tokens without generating
    response = openai_client.responses.create(
        model="gpt-4o-mini",
        input=messages,
    )
    
    tokens = response.usage.input_tokens
    tokens_per_page_api.append(tokens)
    print(f"{doc['filename']}: {tokens} input tokens")

avg_tokens_api = sum(tokens_per_page_api) / len(tokens_per_page_api)
print(f"\nAverage input tokens (OpenAI API): {avg_tokens_api:.0f}")

01-agentic-rag/lessons/01-intro.md: 871 input tokens
01-agentic-rag/lessons/02-environment.md: 1085 input tokens
01-agentic-rag/lessons/03-rag.md: 1497 input tokens

Average input tokens (OpenAI API): 1151


Q2: 2. After running text_search for the first ground truth question, what is the filename of the first result? (1 point) ANS: 01-agentic-rag/lessons/01-intro.md

In [10]:
import pandas as pd
from ingest import load_faq_data, build_index

# 1. Load and prepare documents
documents = load_faq_data()

documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)
documents = documents_llm

# 2. Create text search index (same as in your previous module)
index = build_index(documents)


# 3. Load ground truth data
df_ground_truth = pd.read_csv("data/ground_truth.csv")

# 4. Get the first question
first_question = df_ground_truth.iloc[0]["question"]
print(f"First ground truth question: {first_question}")
print()

# 5. Run text search with boost (using your previous parameters)
boost = {'question': 3.0}  # Boost question field for better matching
text_results = index.search(
    first_question,
    num_results=5,
    boost_dict=boost
)

# 6. Get the answer
first_result = text_results[0]
print(f"Q2 Answer: {first_result}")

First ground truth question: I just found this course late — can I still join now?

Q2 Answer: {'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}


Q3: After running vector_search for the same question, what is the filename of the first result? (1 point) ANS: 

In [28]:
import numpy as np
import pandas as pd
from embedder import Embedder
from minsearch import VectorSearch
from ingest import load_faq_data

# 1. Initialize ONNX embedder
embed = Embedder()

# 2. Load and prepare documents
documents = load_faq_data()

documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)
documents = documents_llm

# 3. Create embeddings
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

batch_size = 50
vectors = []

for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)

# 4. Create vector search index
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

# 5. Load ground truth
df_ground_truth = pd.read_csv("data/ground_truth.csv")

# 6. Get first question
first_question = df_ground_truth.iloc[0]["question"]

# 7. Vector search
query_vector = embed.encode(first_question)
vector_results = vindex.search(query_vector, num_results=5)

# 8. Answer
print(f"Q3 ANSWER: {vector_results[0]['question']}")

Q3 ANSWER: I just discovered the course. Can I still join?


4. After evaluating text_search on the ground truth, what is the Hit Rate? (1 point) ANS: ~0.88

In [12]:
import pandas as pd
from ingest import load_faq_data, build_index
from tqdm.auto import tqdm

# 1. Load and prepare documents
documents = load_faq_data()

documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm

# 2. Build text search index
index = build_index(documents)

# 3. Define text search function (with boost)
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

# 4. Load ground truth
df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

# 5. Define relevance computation functions
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])
    
    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))
    
    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    
    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)
    
    return relevance_total

# 6. Define hit rate function
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:  # Correct document appears in top 5
            cnt = cnt + 1
    return cnt / len(relevance)

# 7. Calculate hit rate for text search
relevance_total = compute_relevance_total(ground_truth, text_search)
hit_rate_value = hit_rate(relevance_total)

  0%|          | 0/515 [00:00<?, ?it/s]

In [13]:
hit_rate_value

0.8660194174757282

Q5. After evaluating vector_search on the ground truth, what is the MRR? (1 point) ANS: ~

In [34]:
import pandas as pd
import numpy as np
from embedder import Embedder
from minsearch import VectorSearch
from ingest import load_faq_data
from tqdm.auto import tqdm

# 1. Initialize ONNX embedder
embed = Embedder()

# 2. Load and prepare documents
documents = load_faq_data()

documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)
documents = documents_llm
print(f"Loaded {len(documents)} documents")

# 3. Create embeddings for vector search
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)
print(f"Created {X.shape[0]} embeddings of dimension {X.shape[1]}")

# 4. Create vector search index
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)
print("Vector index created!")

# 5. Define vector search function
def vector_search(query):
    query_vector = embed.encode(query)
    return vindex.search(query_vector, num_results=5)

# 6. Load ground truth
df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
print(f"Loaded {len(ground_truth)} ground truth records")

# 7. Compute relevance for a single query
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])
    
    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))
    
    return relevance

# 8. Compute relevance for all queries
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    
    for q in tqdm(ground_truth, desc="Computing relevance"):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)
    
    return relevance_total

# 9. Calculate Hit Rate
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt += 1
    return cnt / len(relevance)

# 10. Calculate MRR (Mean Reciprocal Rank)
def mrr(relevance):
    total_score = 0.0
    
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score += score
                break
    
    return total_score / len(relevance)

# 11. Evaluate function
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

# 12. Run evaluation for vector search
vector_results = evaluate(ground_truth, vector_search)

Loaded 103 documents


  0%|          | 0/3 [00:00<?, ?it/s]

Created 103 embeddings of dimension 384
Vector index created!
Loaded 515 ground truth records


Computing relevance:   0%|          | 0/515 [00:00<?, ?it/s]

In [35]:
vector_results

{'hit_rate': 0.9786407766990292, 'mrr': 0.9008737864077668}

Q6. When evaluating hybrid_search for k values 1, 50, 100, and 200, which k gives the best MRR? (1 point) ANS: 


In [36]:
import pandas as pd
import numpy as np
from embedder import Embedder
from minsearch import VectorSearch
from ingest import load_faq_data, build_index
from tqdm.auto import tqdm

# 1. Initialize ONNX embedder
embed = Embedder()

# 2. Load and prepare documents
documents = load_faq_data()

documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)
documents = documents_llm
print(f"Loaded {len(documents)} documents")

# 3. Build text search index (using build_index from ingest)
index = build_index(documents)
print("Text index built!")

# 4. Create embeddings for vector search
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)
print(f"Created {X.shape[0]} embeddings of dimension {X.shape[1]}")

# 5. Create vector search index
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)
print("Vector index created!")

# 6. Define search functions
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    return index.search(query, num_results=5, boost_dict=boost_dict)

def vector_search(query):
    query_vector = embed.encode(query)
    return vindex.search(query_vector, num_results=5)

# 7. Define hybrid search with RRF
def hybrid_search(query, k=60):
    # Get results from both methods
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # RRF fusion
    scores = {}
    docs = {}
    
    # Process text results (rank starts at 0)
    for rank, doc in enumerate(text_results):
        doc_id = doc['id']
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
        docs[doc_id] = doc
    
    # Process vector results (rank starts at 0)
    for rank, doc in enumerate(vector_results):
        doc_id = doc['id']
        scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
        docs[doc_id] = doc
    
    # Sort by score (highest first)
    sorted_results = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [docs[doc_id] for doc_id, _ in sorted_results[:5]]

# 8. Load ground truth
df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
print(f"Loaded {len(ground_truth)} ground truth records")

# 9. Helper functions for evaluation
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(q["question"])
    
    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))
    
    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth, desc="Computing relevance"):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)
    return relevance_total

def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance)

def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {"mrr": mrr(relevance_total)}

# 10. Test different k values
k_values = [1, 50, 100, 200]
mrr_results = {}

print("\n" + "=" * 50)
print("TESTING HYBRID SEARCH WITH DIFFERENT k VALUES")
print("=" * 50)

for k in k_values:
    # Create a search function with this k value
    def search_with_k(query, k=k):
        return hybrid_search(query, k)
    
    print(f"\nTesting k={k}...")
    results = evaluate(ground_truth, search_with_k)
    mrr_results[k] = results["mrr"]
    print(f"  MRR: {results['mrr']:.4f}")

# 11. Find the best k
best_k = max(mrr_results, key=mrr_results.get)
best_mrr = mrr_results[best_k]

Loaded 103 documents
Text index built!


  0%|          | 0/3 [00:00<?, ?it/s]

Created 103 embeddings of dimension 384
Vector index created!
Loaded 515 ground truth records

TESTING HYBRID SEARCH WITH DIFFERENT k VALUES

Testing k=1...


Computing relevance:   0%|          | 0/515 [00:00<?, ?it/s]

  MRR: 0.8506

Testing k=50...


...

  MRR: 0.8457

Testing k=100...


Computing relevance:   0%|          | 0/515 [00:00<?, ?it/s]

  MRR: 0.8457

Testing k=200...


Computing relevance:   0%|          | 0/515 [00:00<?, ?it/s]

  MRR: 0.8457
